## 💼 Análise Exploratória de Dados (AED) aplicada ao varejo.

#### Carregamento das Bibliotecas:

In [66]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # Mostra todas as colunas
pd.set_option('display.float_format', '{:.2f}'.format)  # Foramatação de String
print('Bibliotecas carregadas com sucesso!')

Bibliotecas carregadas com sucesso!


#### Carregamento e visualização da base de dados:

In [67]:
df_varejo = pd.read_csv('../data/base_varejo.csv', sep=';', encoding='utf-8') 
(df_varejo.head(10))

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN
5,01/02/2019,1000,534,M,4,1,C,187,HIGIENE,HASTES FLEXIVEIS,NaN,NaN,NaN,NaN
6,01/02/2019,1000,534,M,4,1,C,163,ALIMENTOS,MORTADELA,NaN,NaN,NaN,NaN
7,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
8,01/02/2019,1000,534,M,4,1,C,95,LIMPEZA,AMACIANTE,NaN,NaN,NaN,NaN
9,01/02/2019,1000,534,M,4,1,C,198,BEBIDAS,ENERGETICO,NaN,NaN,NaN,NaN


###

#### Diagnóstico:

In [68]:
print('DIAGNÓSTICO ')
print('=' * 55)

print(f'Registros/Linhas:    {df_varejo.shape[0]} \nColunas:             {df_varejo.shape[1]}')
print('\nDuplicatas totais:  ', df_varejo.duplicated().sum())

nulos = df_varejo.isnull().sum()
pct = (nulos / len(df_varejo) * 100).round(1)
print(f'Total de nulos:      {nulos.sum()}')

print(f'\nNulos por coluna:')
if nulos[nulos > 0].empty:
    print('Nenhum nulo encontrado!')
else:
    for col in df_varejo.columns:
        if nulos[col] > 0:
            print(f'  {col}: {nulos[col]} ({pct[col]}%)')

print('\nTipos de Dados:')
print(df_varejo.dtypes)

print('\n' + '=' * 55)
print('CONCLUSÃO: \n4 colunas sem nome e sem nenhum valor!')
print('Existem 96.553 linhas duplicadas.')
print('A coluna "DATA" está com o tipo errado.')
print('=' * 55)

DIAGNÓSTICO 
Registros/Linhas:    830000 
Colunas:             14

Duplicatas totais:   96553
Total de nulos:      3320000

Nulos por coluna:
  Unnamed: 10: 830000 (100.0%)
  Unnamed: 11: 830000 (100.0%)
  Unnamed: 12: 830000 (100.0%)
  Unnamed: 13: 830000 (100.0%)

Tipos de Dados:
DATA               str
CO_ID            int64
CL_ID            int64
CL_GENERO          str
CL_EC            int64
CL_FHL           int64
CL_SEG             str
PR_ID            int64
PR_CAT             str
PR_NOME            str
Unnamed: 10    float64
Unnamed: 11    float64
Unnamed: 12    float64
Unnamed: 13    float64
dtype: object

CONCLUSÃO: 
4 colunas sem nome e sem nenhum valor!
Existem 96.553 linhas duplicadas.
A coluna "DATA" está com o tipo errado.


#### Tratamento de Nulos:

In [69]:
df_clean = df_varejo.drop(columns=['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13'])
df_clean.replace(['NULL', 'N/A', '', ' ',], np.nan, inplace=True) 

categorias_vazias = df_clean['PR_CAT'].isnull().sum()
print(f'Categorias vazias em PR_CAT: {categorias_vazias}')

print(f'\nNulos antes: \n{df_varejo.isnull().sum()}')
print(f'\nNulos após remoção de colunas:')
print(df_clean.isnull().sum())

Categorias vazias em PR_CAT: 0

Nulos antes: 
DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64

Nulos após remoção de colunas:
DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64


Verificação de categorias vazias em ['PR_CAT']

In [70]:
categorias_vazias = df_clean['PR_CAT'].isnull().sum()
print(f'Categorias vazias em PR_CAT: {categorias_vazias}')

for cat in sorted(df_clean['PR_CAT'].unique()):
    print(cat)

Categorias vazias em PR_CAT: 0
#N/D
ACESSORIOS
ALIMENTOS
BEBIDAS
HIGIENE
LIMPEZA
PET


Preencher categorias '#N/D' por 'Sem Categoria' e verificar alterações:

In [71]:
if (df_clean['PR_CAT'] == '#N/D').any():
    df_clean['PR_CAT'] = df_clean['PR_CAT'].replace('#N/D', 'Sem Categoria')
    print('Substituição realizada!')
else:
    print('Substituição não necessária')

print('\nApós substituição:')
for cat in sorted(df_clean['PR_CAT'].unique()):
    print(cat)
print(f'\nTotal de "Sem Categoria": {(df_clean["PR_CAT"] == "Sem Categoria").sum()}')

Substituição realizada!

Após substituição:
ACESSORIOS
ALIMENTOS
BEBIDAS
HIGIENE
LIMPEZA
PET
Sem Categoria

Total de "Sem Categoria": 3650


#### Tratamento de Duplicadas:

In [72]:
print('Duplicatas totais antes:', df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)

print('Duplicatas totais apos limpeza:', df_clean.duplicated().sum())

print(f'\nRegistros/Linhas: {df_clean.shape[0]} \nColunas:          {df_clean.shape[1]}')

Duplicatas totais antes: 96553
Duplicatas totais apos limpeza: 0

Registros/Linhas: 733447 
Colunas:          10


Validação da coluna ['CO_ID']:

In [73]:
invalidos = df_clean[df_clean['CO_ID'] <= 0]
print(f'Registrso com CO_ID inválido: {len(invalidos)}')
print(f'Total de compras únicas: {df_clean["CO_ID"].nunique()}')
print(f'Tipo: {df_clean['CO_ID'].dtype}')

Registrso com CO_ID inválido: 0
Total de compras únicas: 18471
Tipo: int64


#### Transformação de Strings, Integer e Float e Datetime:

In [74]:
df_clean['DATA'] = pd.to_datetime(df_clean['DATA'], format='%d/%m/%Y', errors='coerce')
print('Datas inválidas (NaT):', df_clean['DATA'].isna().sum())
print('Tipo da coluna DATA:', df_clean['DATA'].dtype)

Datas inválidas (NaT): 0
Tipo da coluna DATA: datetime64[us]


#### Estatística Descritiva:

In [90]:
print(df_clean['CL_FHL'].describe())
print(f'Mediana: {df_clean["CL_FHL"].median()}')
print(f'Moda: {df_clean["CL_FHL"].mode()[0]}')

count   733447.00
mean         1.15
std          1.42
min          0.00
25%          0.00
50%          0.00
75%          2.00
max          4.00
Name: CL_FHL, dtype: float64
Mediana: 0.0
Moda: 0


### Padrões de Agrupamento:

Agrupar com 'groupby' as vendas por gênero, para identificar qual o gênero com mais compras:

In [89]:
resultado = df_clean.groupby('CL_GENERO')['CO_ID'].count()

print('\nCompras por Gênero:')
for genero, total in resultado.items():
    print(f'{genero}= {total:,}')


Compras por Gênero:
F= 382,427
M= 351,020


Agrupar com 'pivot_table' as vendas por categoria, para identificar qual a categoria com mais vendas.

In [78]:
pd.pivot_table(
    df_clean,
    values='CO_ID',
    index='PR_CAT',
    aggfunc='count'
).sort_values('CO_ID', ascending=False)

,CO_ID
PR_CAT,
ALIMENTOS,384197
HIGIENE,137702
LIMPEZA,128632
BEBIDAS,38264
PET,28553
ACESSORIOS,12871
Sem Categoria,3228


Agrupar com 'pivot_table' as vendas por produto, para identificar qual o produto com mais vendas.

In [79]:
pd.pivot_table(
    df_clean,
    values='CO_ID',
    index='PR_NOME',
    aggfunc='count'
).sort_values('CO_ID', ascending=False)

,CO_ID
PR_NOME,
PRESUNTO COZIDO,12719
SARDINHA,6610
BANANA,6518
ESCOVA DE DENTE,6518
GEL,6517
...,...
#N/D,3228
ABSORVENTE,3220
ALIMENTO PARA PASSARO,3198


Agrupar com 'pivot_table' as vendas por categoria, de acordo com o GÊNERO, para identificar se há diferença de consumo entre homens e mulheres.

In [82]:
pd.pivot_table(
    df_clean,
    values='CO_ID',
    index='PR_CAT',
    columns='CL_GENERO',
    aggfunc='count'
).sort_values('F', ascending=False)

CL_GENERO,F,M
PR_CAT,,
ALIMENTOS,200274,183923
HIGIENE,71721,65981
LIMPEZA,67328,61304
BEBIDAS,19764,18500
PET,14809,13744
ACESSORIOS,6839,6032
Sem Categoria,1692,1536


Criação de uma coluna 'TEM_FILHO' em um df separado, para poder investigar se existe alteração no hábito de consumo entre clientes com filho e sem. Exibição total por cliente com e sem filho e agrupamneto por categoria: 

In [86]:
df_analise = df_clean.copy()
df_analise['TEM_FILHO'] = df_analise['CL_FHL'].apply(lambda x: 'Com filhos' if x > 0 else 'Sem filhos')

resultado = df_analise['TEM_FILHO'].value_counts()
for grupo, total in resultado.items():
    print(f'{grupo}: {total:,}')
    
pd.pivot_table(
    df_analise,
    values='CO_ID',
    index='PR_CAT',
    columns='TEM_FILHO',
    aggfunc='count'
).sort_values('Com filhos', ascending=False)


Sem filhos: 384,986
Com filhos: 348,461


TEM_FILHO,Com filhos,Sem filhos
PR_CAT,,
ALIMENTOS,182527,201670
HIGIENE,65400,72302
LIMPEZA,61007,67625
BEBIDAS,18273,19991
PET,13593,14960
ACESSORIOS,6116,6755
Sem Categoria,1545,1683


Salvamento do df_clean na pasta 'data':

In [91]:
df_clean.to_csv('../data/df_limpo.csv', index=False, sep=';', encoding='utf-8')
print('Base limpa salva com sucesso!')

Base limpa salva com sucesso!
